# Native Sparse Attention：压缩、选择与局部窗口协同

**面试问题：原生稀疏注意力怎样在长文档中既降低计算又保住远距离证据？**

## 回答主线

1. 稀疏注意力不能简单等同于固定局部窗口，因为远距离政策和实体常决定当前答案。
2. Native Sparse Attention 将路径拆成压缩摘要、按查询选择的细粒度块和最近局部窗口。
3. 压缩分支负责低成本扫描全局，选择分支把预算集中到高相关块，局部分支维护邻近语法。
4. 稀疏输出应与 dense teacher 在同一查询上比较误差和保留注意力质量。
5. 块摘要可能稀释块内唯一关键 Token，所以摘要器与块边界本身也是模型质量的一部分。
6. 生产实现还必须让逻辑稀疏真正变成规整 Kernel，否则 FLOPs 下降不等于延迟下降。

## 真实案例

一份售后手册被切成八个双 Token 块，查询是“已签收的耳机多久可以退”。退款期限在文档前部，当前问题在末尾。我们以可读二维 key/value 表示语义，比较最近窗口、dense attention 和“压缩扫描 + Top-K 块 + 局部窗口”的稀疏实现。教学实验使用可读的小数据解释机制，结果不能外推为线上收益。

### 输入预览：八个文档块与查询

In [1]:
import numpy as np  # 导入数组运算以实现 dense 与稀疏注意力。

blocks = [  # 构造八个具有真实售后语义的文档块。
    ["退货期限", "签收7天"],  # 放置需要远距离召回的核心政策。
    ["商品条件", "包装完整"],  # 放置退货条件。
    ["例外商品", "定制除外"],  # 放置例外规则。
    ["退款路径", "原路退回"],  # 放置资金路径。
    ["物流费用", "质量问题免邮"],  # 放置运费规则。
    ["审核材料", "订单照片"],  # 放置申请材料。
    ["处理时效", "仓库2天"],  # 放置内部处理时效。
    ["用户问题", "耳机已签收"],  # 放置当前局部上下文。
]  # 完成文档分块。
token_names = [token for block in blocks for token in block]  # 展开为十六个 Token 名称。
keys = np.array([  # 用退货相关性和耳机/签收相关性表示每个 Token 的 key。
    [1.0, 0.2], [1.0, 0.8], [0.6, 0.2], [0.5, 0.1], [0.2, 0.5], [0.1, 0.2], [0.7, 0.1], [0.4, 0.1],  # 定义前四块 key。
    [0.3, 0.1], [0.2, 0.2], [0.4, 0.2], [0.3, 0.2], [0.3, 0.1], [0.2, 0.1], [0.2, 0.4], [0.5, 1.0],  # 定义后四块 key。
], dtype=float)  # 完成 key 矩阵。
values = np.array([[7.0, 1.0], [7.0, 1.0], [1.0, 0.2], [1.0, 0.1], [0.0, 0.5], [0.0, 0.2], [2.0, 0.5], [0.0, 0.1], [0.0, 0.2], [0.0, 0.2], [0.0, 0.3], [0.0, 0.2], [2.0, 0.2], [2.0, 0.2], [0.0, 0.5], [0.0, 1.0]], dtype=float)  # 用第一维政策天数、第二维证据强度表示 value。
query = np.array([1.0, 0.9])  # 构造同时关注退货和耳机签收的查询向量。
print(f"块数={len(blocks)}，Token 数={len(token_names)}，query={query}")  # 输出输入规模。
for block_id, block in enumerate(blocks):  # 逐块展示真实文档内容。
    print(f"block={block_id} tokens={block} keys={np.round(keys[block_id * 2:block_id * 2 + 2], 2).tolist()}")  # 展示块与 key 的对应关系。

块数=8，Token 数=16，query=[1.  0.9]
block=0 tokens=['退货期限', '签收7天'] keys=[[1.0, 0.2], [1.0, 0.8]]
block=1 tokens=['商品条件', '包装完整'] keys=[[0.6, 0.2], [0.5, 0.1]]
block=2 tokens=['例外商品', '定制除外'] keys=[[0.2, 0.5], [0.1, 0.2]]
block=3 tokens=['退款路径', '原路退回'] keys=[[0.7, 0.1], [0.4, 0.1]]
block=4 tokens=['物流费用', '质量问题免邮'] keys=[[0.3, 0.1], [0.2, 0.2]]
block=5 tokens=['审核材料', '订单照片'] keys=[[0.4, 0.2], [0.3, 0.2]]
block=6 tokens=['处理时效', '仓库2天'] keys=[[0.3, 0.1], [0.2, 0.1]]
block=7 tokens=['用户问题', '耳机已签收'] keys=[[0.2, 0.4], [0.5, 1.0]]


## Baseline 基线：只看最近四个 Token

In [2]:
def softmax(scores):  # 实现数值稳定的 softmax 权重。
    shifted = scores - np.max(scores)  # 减去最大分数避免指数溢出。
    weights = np.exp(shifted)  # 把相似度转换为正权重。
    return weights / weights.sum()  # 返回归一化注意力。

def attend(selected_indices):  # 在指定 Token 子集上计算注意力输出。
    scores = (keys[selected_indices] @ query) * 2.5  # 使用固定温度放大相关性差异并计算查询与选中 key 的点积。
    weights = softmax(scores)  # 在可见子集内归一化。
    output = weights @ values[selected_indices]  # 聚合选中 Token 的 value。
    return output, weights, scores  # 返回输出、权重和原始分数。

local_indices = np.arange(len(token_names) - 4, len(token_names))  # 只保留最后两个文档块。
local_output, local_weights, local_scores = attend(local_indices)  # 运行固定局部窗口基线。
print("局部窗口可见 Token：", [token_names[index] for index in local_indices])  # 展示远距离政策已不可见。
print("局部注意力权重：", np.round(local_weights, 3).tolist())  # 展示局部归一化结果。
print(f"局部输出=[政策天数={local_output[0]:.3f}, 证据强度={local_output[1]:.3f}]")  # 展示错误接近零天的政策信号。

局部窗口可见 Token： ['处理时效', '仓库2天', '用户问题', '耳机已签收']
局部注意力权重： [0.063, 0.049, 0.097, 0.791]
局部输出=[政策天数=0.225, 证据强度=0.862]


### 核心实现：全局压缩扫描、Top-K 块选择与局部并集

In [3]:
dense_indices = np.arange(len(token_names))  # 定义完整 dense teacher 的可见范围。
dense_output, dense_weights, dense_scores = attend(dense_indices)  # 计算完整注意力作为质量参照。
block_summaries = keys.reshape(len(blocks), 2, 2).mean(axis=1)  # 用每块 key 均值形成低成本压缩表示。
block_scores = block_summaries @ query  # 用压缩摘要扫描全部八块。
selected_blocks = np.argsort(-block_scores, kind="stable")[:2]  # 选择查询最相关的两个远近块。
local_blocks = np.array([6, 7])  # 固定保留最近两个块维持局部信息。
active_blocks = np.unique(np.concatenate([selected_blocks, local_blocks]))  # 合并选择分支与局部分支并去重。
sparse_indices = np.concatenate([np.arange(block * 2, block * 2 + 2) for block in active_blocks])  # 展开为真正参与细粒度注意力的 Token 索引。
sparse_output, sparse_weights, sparse_scores = attend(sparse_indices)  # 在稀疏 Token 集合上执行精确注意力。
print("压缩块分数：")  # 输出全局扫描的中间量。
for block_id, score in enumerate(block_scores):  # 逐块展示摘要相关性和选择结果。
    print(f"block={block_id} score={score:.3f} selected={block_id in set(active_blocks)} text={'/'.join(blocks[block_id])}")  # 让 Top-K 决策可解释。
print("稀疏可见 Token：", [token_names[index] for index in sparse_indices])  # 展示远距离政策和最近问题同时存在。

压缩块分数：
block=0 score=1.450 selected=True text=退货期限/签收7天
block=1 score=0.685 selected=False text=商品条件/包装完整
block=2 score=0.465 selected=False text=例外商品/定制除外
block=3 score=0.640 selected=False text=退款路径/原路退回
block=4 score=0.385 selected=False text=物流费用/质量问题免邮
block=5 score=0.530 selected=False text=审核材料/订单照片
block=6 score=0.340 selected=True text=处理时效/仓库2天
block=7 score=0.980 selected=True text=用户问题/耳机已签收
稀疏可见 Token： ['退货期限', '签收7天', '处理时效', '仓库2天', '用户问题', '耳机已签收']


## 结果解读：计算 Token 数、Dense 误差与保留质量

In [4]:
sparse_error = float(np.linalg.norm(sparse_output - dense_output))  # 计算稀疏输出相对 dense teacher 的二范数误差。
local_error = float(np.linalg.norm(local_output - dense_output))  # 计算纯局部基线误差。
dense_mass_kept = float(dense_weights[sparse_indices].sum())  # 计算稀疏集合覆盖的 dense 注意力质量。
print("方案        参与Token  政策天数  证据强度  对Dense误差")  # 输出同一查询下的结果表头。
print(f"Local      {len(local_indices):>9} {local_output[0]:>8.3f} {local_output[1]:>9.3f} {local_error:>11.3f}")  # 展示局部窗口基线。
print(f"Sparse     {len(sparse_indices):>9} {sparse_output[0]:>8.3f} {sparse_output[1]:>9.3f} {sparse_error:>11.3f}")  # 展示原生稀疏结果。
print(f"Dense      {len(dense_indices):>9} {dense_output[0]:>8.3f} {dense_output[1]:>9.3f} {0.0:>11.3f}")  # 展示完整参照。
print(f"稀疏集合覆盖 dense 注意力质量={dense_mass_kept:.1%}，Token 计算减少={1.0 - len(sparse_indices) / len(dense_indices):.1%}")  # 同时报告质量与计算。
print("解读：压缩扫描找到 block0，细粒度分支读取“签收7天”；局部窗口单独无法恢复这个远距离政策。")  # 解释机制而不是只报数值。

方案        参与Token  政策天数  证据强度  对Dense误差
Local              4    0.225     0.862       3.653
Sparse             6    4.893     0.957       1.028
Dense             16    3.878     0.796       0.000
稀疏集合覆盖 dense 注意力质量=76.3%，Token 计算减少=62.5%
解读：压缩扫描找到 block0，细粒度分支读取“签收7天”；局部窗口单独无法恢复这个远距离政策。


## 失败案例：均值摘要稀释块内唯一关键 Token

In [5]:
rare_keys = np.array([[2.0, 1.5], [-2.0, -1.5]])  # 构造一个关键 Token 与无关 Token 恰好相互抵消的块。
mean_summary = rare_keys.mean(axis=0)  # 使用均值压缩会得到全零摘要。
mean_score = float(mean_summary @ query)  # 计算被稀释后的块分数。
token_scores = rare_keys @ query  # 计算块内每个 Token 的真实相关性。
max_score = float(token_scores.max())  # 用最大相关性摘要保留稀有强证据。
selected_by_mean = mean_score > 1.0  # 模拟均值摘要的选择门槛。
selected_by_max = max_score > 1.0  # 模拟查询相关最大池化的选择门槛。
print(f"稀有块 token 分数={np.round(token_scores, 3).tolist()}，均值摘要分数={mean_score:.3f}，最大池化分数={max_score:.3f}")  # 展示信息抵消过程。
print(f"均值摘要是否选择={selected_by_mean}，查询相关最大池化是否选择={selected_by_max}")  # 展示失败和修正结果。
print("修正策略：训练可查询的压缩器，或保留 mean/max 多统计量；还要让关键实体跨块边界重复覆盖。")  # 给出可实施修正。

稀有块 token 分数=[3.35, -3.35]，均值摘要分数=0.000，最大池化分数=3.350
均值摘要是否选择=False，查询相关最大池化是否选择=True
修正策略：训练可查询的压缩器，或保留 mean/max 多统计量；还要让关键实体跨块边界重复覆盖。


### 生产边界与稀疏模式账本

In [6]:
sparsity_ledger = {"query": "已签收耳机多久可退", "selected_blocks": active_blocks.tolist(), "local_blocks": local_blocks.tolist(), "active_tokens": len(sparse_indices), "dense_tokens": len(dense_indices), "dense_mass_kept": round(dense_mass_kept, 4)}  # 构造请求级稀疏决策账本。
print("稀疏账本：", sparsity_ledger)  # 展示线上定位漏召回所需字段。
print("生产替换点：真实 NSA 需要可训练压缩卷积、因果块选择、GQA 头布局、块稀疏 Kernel、反向传播和长文本基准。")  # 明确二维教学实现的边界。

稀疏账本： {'query': '已签收耳机多久可退', 'selected_blocks': [0, 6, 7], 'local_blocks': [6, 7], 'active_tokens': 6, 'dense_tokens': 16, 'dense_mass_kept': 0.7626}
生产替换点：真实 NSA 需要可训练压缩卷积、因果块选择、GQA 头布局、块稀疏 Kernel、反向传播和长文本基准。


## 回归测试：最后只保护选择、质量与摘要反例

In [7]:
assert 0 in set(active_blocks) and 7 in set(active_blocks)  # 验证远距离政策块和最近问题块同时可见。
assert len(sparse_indices) < len(dense_indices)  # 验证稀疏路径减少参与注意力的 Token 数。
assert sparse_error < local_error  # 验证同一查询下稀疏输出比纯局部更接近 dense teacher。
assert dense_mass_kept > 0.55  # 验证所选 Token 覆盖多数 dense 注意力质量。
assert not selected_by_mean and selected_by_max  # 验证均值稀释反例和最大池化修正均实际发生。
print("回归测试通过：远距召回、局部并集、计算节省、Dense 近似与摘要失败修正均成立。")  # 用少量断言总结稀疏注意力合同。

回归测试通过：远距召回、局部并集、计算节省、Dense 近似与摘要失败修正均成立。
